# Notebook 04 - Data Preparation & Target Generation

**Purpose:** transform the raw CSV into model-ready tensors. Produce the
analytically-derived targets, fit the scaler on training data only, and
persist everything to Drive.

**Inputs:** `full_pixels_10m.csv`, `selected_features.json`
**Outputs:** `X_scaled.npy`, `y_burned.npy`, `targets_aux.npz`, `splits.json`,
`scaler.pkl`, `feature_names.json`, padding mask, and pixel metadata


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


Mounted at /content/drive


In [2]:
# Load shared configuration.
# config.py must sit at /content/drive/MyDrive/thesis/config.py
import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/thesis")
import config
importlib.reload(config)

config.ensure_dirs()
config.set_seed()
config.print_summary()


Version:       _10m
GEE project:   bar-project-457815
Scale:         10 m   CRS: EPSG:32636
Period:        2017-2025  (107 months)
S2 collection: projects/bar-project-457815/assets/s2_monthly_image_collection_10m
Features:      27 model inputs (28 total available)
RPS weights:   {'recurrence': 0.25, 'severity': 0.25, 'persistence': 0.25, 'recovery': 0.25}
Seed:          42


In [3]:
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


## 1. Load raw data and selected features

In [4]:
df = pd.read_csv(config.FILES["raw_timeseries"])
df = df.sort_values(["pixel_id", "year", "month"]).reset_index(drop=True)

with open(config.FILES["selected_features"]) as f:
    selected_features = json.load(f)["selected"]

print(f"Pixels:   {df['pixel_id'].nunique()}")
print(f"Features: {len(selected_features)}")


Pixels:   9667
Features: 26


## 2. Reshape to [N, T, C]

Each pixel becomes a fixed-length sequence of `EXPECTED_T` months. Padding
is applied if a pixel has fewer rows. A boolean mask records which positions
are real data and which are padding - the model uses this mask in its
attention.


In [5]:
month_list  = config.get_month_list()
month_index = {(y, m): i for i, (y, m) in enumerate(month_list)}
T = config.EXPECTED_T

pixel_ids = df["pixel_id"].unique()
N = len(pixel_ids)
C = len(selected_features)

X    = np.full((N, T, C), np.nan, dtype=np.float32)
y    = np.zeros((N, T), dtype=np.float32)
mask = np.zeros((N, T), dtype=bool)
meta = []

for i, pid in enumerate(pixel_ids):
    g = df[df["pixel_id"] == pid]
    for _, row in g.iterrows():
        key = (int(row["year"]), int(row["month"]))
        if key not in month_index:
            continue
        t = month_index[key]
        X[i, t, :] = row[selected_features].values
        y[i, t]    = row["burned"]
        mask[i, t] = True

    meta.append({
        "pixel_id":   int(pid),
        "pixel_type": g["pixel_type"].iloc[0],
    })

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"mask coverage: {100 * mask.mean():.1f}%")


X shape: (9667, 107, 26)
y shape: (9667, 107)
mask coverage: 96.0%


## 3. NaN imputation by calendar month

Replace NaN with the mean for each (calendar_month, feature) pair. Respects
seasonality: a missing July value gets the July mean, not a global mean that
would mix winter and summer.


In [6]:
calendar_month = np.array([m for _, m in month_list])

for cm in range(1, 13):
    t_idx = np.where(calendar_month == cm)[0]
    if len(t_idx) == 0:
        continue
    block = X[:, t_idx, :]
    flat  = block.reshape(-1, C)
    col_means = np.nanmean(flat, axis=0)
    inds = np.where(np.isnan(flat))
    flat[inds] = np.take(col_means, inds[1])
    X[:, t_idx, :] = flat.reshape(block.shape)

# Fallback: any remaining NaNs (entire calendar month missing for a feature)
if np.isnan(X).any():
    global_means = np.nanmean(X.reshape(-1, C), axis=0)
    inds = np.where(np.isnan(X.reshape(-1, C)))
    flat = X.reshape(-1, C)
    flat[inds] = np.take(global_means, inds[1])
    X = flat.reshape(N, T, C)

assert not np.isnan(X).any(), "NaN imputation incomplete"
print("Imputation complete; no NaN remain.")


Imputation complete; no NaN remain.


## 4. Compute analytical targets

Severity, persistence, recovery, and the composite RPS are derived from the
NBR z-score time series. These are *analytical* targets - they are not field
measurements. The model learns to predict them, but their ecological validity
rests on the formulas defined here.


In [7]:
def compute_targets(X, y, feature_names, rps_weights=None):
    """Compute auxiliary regression targets.

    severity:    mean negative NBR z-score during fire months (0-1)
    persistence: fraction of post-fire months with NBR z-score below -0.5
    recovery:    slope of NBR z-score after the last fire (rescaled to 0-1)
    recurrence:  fire count normalised by 5 (capped at 1)
    rps:         weighted combination, gated to 0 for never-burned pixels
    """
    if rps_weights is None:
        w = config.RPS_WEIGHTS
        rps_weights = (w["recurrence"], w["severity"],
                       w["persistence"], w["recovery"])

    N, T, C = X.shape
    try:
        nbr_z_idx = feature_names.index("NBR_zscore")
    except ValueError:
        nbr_z_idx = feature_names.index("NBR")

    severity    = np.zeros(N, dtype=np.float32)
    persistence = np.zeros(N, dtype=np.float32)
    recovery    = np.zeros(N, dtype=np.float32)
    recurrence  = np.zeros(N, dtype=np.float32)
    has_burned  = np.zeros(N, dtype=np.float32)

    for i in range(N):
        nbr_z         = X[i, :, nbr_z_idx]
        burned_months = np.where(y[i] == 1)[0]

        if len(burned_months) == 0:
            continue

        has_burned[i] = 1.0
        recurrence[i] = float(np.clip(len(burned_months) / 5.0, 0, 1))

        drops          = -nbr_z[burned_months]
        severity[i]    = float(np.clip(drops.mean() / 5.0, 0, 1))

        last_fire = burned_months[-1]
        post_fire = nbr_z[last_fire:]
        if len(post_fire) > 1:
            persistence[i] = float((post_fire < -0.5).mean())
        if len(post_fire) > 3:
            t_axis = np.arange(len(post_fire), dtype=np.float32)
            slope  = np.polyfit(t_axis, post_fire, 1)[0]
            recovery[i] = float(np.clip((slope + 0.1) / 0.2, 0, 1))

    w1, w2, w3, w4 = rps_weights
    rps = has_burned * (
        w1 * recurrence + w2 * severity +
        w3 * persistence + w4 * (1 - recovery)
    )
    rps = np.clip(rps, 0, 1).astype(np.float32)

    return {
        "recurrence":  recurrence,
        "severity":    severity,
        "persistence": persistence,
        "recovery":    recovery,
        "rps":         rps,
    }


targets = compute_targets(X, y, selected_features)

print("Target statistics (mean / std / max):")
for name, arr in targets.items():
    print(f"  {name:12s}  {arr.mean():.3f}  {arr.std():.3f}  {arr.max():.3f}")


Target statistics (mean / std / max):
  recurrence    0.167  0.219  1.000
  severity      0.099  0.201  1.000
  persistence   0.212  0.252  1.000
  recovery      0.222  0.327  1.000
  rps           0.197  0.200  0.872


## 5. Train / validation / test split

Random stratified split by pixel type. Stratification ensures each split
contains some of every burn-recurrence category. Non-random splits
(temporal, spatial) are explored in Notebook 06.


In [8]:
meta_df = pd.DataFrame(meta)
strat = meta_df["pixel_type"].apply(
    lambda t: "burned" if "burned" in t else "never_burned"
)

idx_all = np.arange(N)
idx_train, idx_temp = train_test_split(
    idx_all, test_size=1 - config.SPLITS["train_frac"],
    stratify=strat, random_state=config.SEED,
)
val_size = config.SPLITS["val_frac"] / (
    config.SPLITS["val_frac"] + config.SPLITS["test_frac"]
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=1 - val_size,
    stratify=strat.iloc[idx_temp], random_state=config.SEED,
)

print(f"Train: {len(idx_train)}   Val: {len(idx_val)}   Test: {len(idx_test)}")
for name, ix in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    print(f"  {name}: {meta_df.iloc[ix]['pixel_type'].value_counts().to_dict()}")


Train: 6766   Val: 1450   Test: 1451
  train: {'never_burned': 3125, 'burned_low': 2951, 'burned_medium': 549, 'burned_high': 141}
  val: {'never_burned': 677, 'burned_low': 624, 'burned_medium': 120, 'burned_high': 29}
  test: {'never_burned': 698, 'burned_low': 619, 'burned_medium': 104, 'burned_high': 30}


## 6. Scale features (fit on train only)

`StandardScaler` is fit on training data alone, then applied to validation
and test. Prevents leakage of test-set statistics into the training pipeline.


In [9]:
X_train_flat = X[idx_train].reshape(-1, C)
scaler = StandardScaler().fit(X_train_flat)

X_scaled = (scaler.transform(X.reshape(-1, C))
                  .reshape(N, T, C).astype(np.float32))
print(f"X_scaled shape: {X_scaled.shape}")
print(f"X_scaled mean ~0: {X_scaled.mean():.3f}, std ~1: {X_scaled.std():.3f}")


X_scaled shape: (9667, 107, 26)
X_scaled mean ~0: -0.000, std ~1: 1.018


## 7. Persist everything

In [10]:
np.save(config.FILES["X_scaled"], X_scaled)
np.save(config.FILES["y_burned"], y)
np.savez(config.FILES["targets_aux"], **targets)
np.save(config.FILES["X_scaled"].replace(".npy", "_mask.npy"), mask)

with open(config.FILES["splits"], "w") as f:
    json.dump({
        "train": idx_train.tolist(),
        "val":   idx_val.tolist(),
        "test":  idx_test.tolist(),
    }, f)

with open(config.FILES["scaler"], "wb") as f:
    pickle.dump(scaler, f)

with open(config.FILES["feature_names"], "w") as f:
    json.dump(selected_features, f)

meta_df.to_csv(config.FILES["X_scaled"].replace(".npy", "_meta.csv"),
               index=False)

print("All artefacts written to", config.PATHS["data_processed"])


All artefacts written to /content/drive/MyDrive/thesis/data/processed
